# 34. Sparse SCF migration (`SparseMigration`)

**Objectives:**
- Build the same Gaussian-smearing migration map as a `SparseMigration`
  (COO representation) instead of a dense matrix.
- Compare `nnz` to `n_bins**2` to see the memory saving the docstring
  describes ("O(nnz) memory instead of O(n_bins^2)").
- Confirm the sparse and dense forms give identical `smeared_density_at`/
  `scf_fraction_at` results on a grid small enough to build both.

Run cells top to bottom in a fresh kernel. Masses are in GeV, invariants in GeV²,
daughter indices start at zero. This continues from
[tutorial 33](tutorial_33_scf_map_dense.ipynb); see `docs/scf.md`.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes use complex128.

import numpy as np
import jax.numpy as jnp
from dalitzplotfitter import (
    DecayChannel, DecayModel, NonResonant, RealImag, Resonance,
    SparseMigration, SquareDalitzSCFMap, generate_toy,
)

## 1. Same model and dense migration matrix as tutorial 33

We rebuild the tridiagonal-in-flattened-index smearing kernel exactly as
before, on the same `N = 10` grid, so the dense and sparse maps below are
directly comparable.

In [2]:
channel = DecayChannel("B+", ("K+", "pi+", "pi-"))
model = DecayModel(
    channel,
    [
        Resonance("Kstar", (0, 2), RealImag(1.0, 0.0), mass=0.892, width=0.051, spin=1),
        NonResonant(RealImag(0.4, -0.2), name="NR"),
    ],
    normalization_method="square-dalitz", normalization_pair=(0, 2),
    normalization_resolution=60,
)
truth = {p.name: p.value for p in model.parameters}

N = 10
n_bins = N * N
migration = np.eye(n_bins) * 0.70
migration += np.roll(np.eye(n_bins), 1, axis=1) * 0.15
migration += np.roll(np.eye(n_bins), -1, axis=1) * 0.15
migration /= migration.sum(axis=1, keepdims=True)
scf_fraction = np.full(n_bins, 0.12)

## 2. `SparseMigration.from_dense` and building `SquareDalitzSCFMap(storage="sparse")`

`SparseMigration.from_dense` compresses a dense matrix once, on the host,
into `(true_indices, reco_indices, probabilities)` COO arrays. Passing it to
`SquareDalitzSCFMap` (or `storage="sparse"` on a dense array) keeps the map's
migration operator sparse for every subsequent evaluation.

In [3]:
sparse_operator = SparseMigration.from_dense(migration)
print(f"n_bins**2 = {n_bins * n_bins}, nnz = {sparse_operator.nnz}, "
      f"density = {sparse_operator.density:.3f}")

sparse_map = SquareDalitzSCFMap(
    migration=sparse_operator,
    scf_fraction=scf_fraction,
    mother_mass=channel.parent_mass,
    masses=channel.daughter_masses,
    bins_mprime=N,
    bins_thetaprime=N,
    pair=(0, 2),
)
dense_map = SquareDalitzSCFMap(
    migration=migration,
    scf_fraction=scf_fraction,
    mother_mass=channel.parent_mass,
    masses=channel.daughter_masses,
    bins_mprime=N,
    bins_thetaprime=N,
    pair=(0, 2),
    storage="dense",
)
assert sparse_map.is_sparse and not dense_map.is_sparse
print(f"Dense storage would need {n_bins * n_bins} float64 entries "
      f"({8 * n_bins * n_bins / 1e3:.1f} kB); "
      f"sparse storage needs {sparse_operator.nnz} ({8 * sparse_operator.nnz / 1e3:.1f} kB).")

n_bins**2 = 10000, nnz = 300, density = 0.030


Dense storage would need 10000 float64 entries (80.0 kB); sparse storage needs 300 (2.4 kB).


## 3. Same results on `smeared_density_at` and `scf_fraction_at`

Both maps describe the same migration, just stored differently, so every
public query must agree to machine precision.

In [4]:
toy = generate_toy(model, 4000, parameters=truth, seed=34, include_momenta=False)
data = toy.as_dict()

true_data = sparse_map.true_bin_data()
true_density = model.intensity(true_data, truth)

sparse_smeared = np.asarray(
    sparse_map.smeared_density_at(true_density, data["s12"], data["s13"], data["s23"])
)
dense_smeared = np.asarray(
    dense_map.smeared_density_at(true_density, data["s12"], data["s13"], data["s23"])
)
np.testing.assert_allclose(sparse_smeared, dense_smeared, rtol=1e-10, atol=1e-12)

sparse_fraction = np.asarray(sparse_map.scf_fraction_at(data["s12"], data["s13"], data["s23"]))
dense_fraction = np.asarray(dense_map.scf_fraction_at(data["s12"], data["s13"], data["s23"]))
np.testing.assert_allclose(sparse_fraction, dense_fraction)

print("Sparse and dense smeared densities agree to machine precision:",
      bool(np.allclose(sparse_smeared, dense_smeared, rtol=1e-10)))

Sparse and dense smeared densities agree to machine precision: True


## Try it yourself

1. Make the migration kernel wider and check `sparse_operator.density` grows.
2. Try `storage="auto"` with `sparse_threshold=0.05` on the same dense matrix
   and confirm whether it picks sparse or dense storage.
3. Time `dense_map.smeared_bin_density` vs. `sparse_map.smeared_bin_density`
   on a much larger `N` (e.g. 40) where only the sparse map is practical.

## Continue learning

Next: [Histogram efficiency/background maps from arrays](tutorial_35_histogram_maps_from_arrays.ipynb).
Reference: [SCF migration](../../docs/scf.md).

Return to [the course guide](TUTORIALS.md).